# Lab 03: Load and investigate multiple datasets (solution)

You will find the course data files, read them with the `csv` module, pandas and Excel, handle missing and unsupported files, and write outputs.

---
# Part A: Finding files

## A1. Set up
`Path` (from `pathlib`) represents a folder or file. `../../data` means *two folders up from this notebook, then into `data`*.

*Run this cell; no changes needed.*

In [ ]:
from pathlib import Path
import csv
import pandas as pd

DATA = Path("../../data")
OUT = Path("output")
print(DATA.resolve())

## A2. List the data files
`DATA.glob("*.*")` finds every file in the folder. Build a list `files` of the file **names** (`f.name`).

**Example**
```python
names = []
for f in DATA.glob("*.csv"):
    names.append(f.name)
```

In [ ]:
files = []
for f in DATA.glob("*.*"):
    files.append(f.name)
print(files)

In [ ]:
# check
assert "trade_summary.csv" in files and "countries.xlsx" in files
print("A2 OK")

## A3. Show file sizes
`f.stat().st_size` is the size in bytes. Print each file's name and size in KB (divide by 1024), to one decimal place.

**Example**
```python
print(f"{name}: {size_kb:.1f} KB")
```

In [ ]:
for f in DATA.glob("*.*"):
    size_kb = f.stat().st_size / 1024
    print(f"{f.name}: {size_kb:.1f} KB")

---
# Part B: Reading a CSV file with the csv module

## B1. Read the file
`open()` opens the file; `with` closes it automatically. `csv.DictReader` turns each row into a dictionary (Module 02), and `list()` collects all rows.

*Run this cell; no changes needed.*

In [ ]:
with open(DATA / "trade_summary.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(len(rows), "rows")
print(rows[0])

## B2. Look inside one row
`rows[0]` is a dictionary. Store its reporter in `reporter` and its exports in `first_exports`, then print `type(first_exports)`.

In [ ]:
first = rows[0]
reporter = first["reporter"]
first_exports = first["exports_usd_m"]
print(reporter, first_exports, type(first_exports))

In [ ]:
# check
assert reporter == "Australia"
assert isinstance(first_exports, str)
print("B2 OK: note that every value from the csv module is text")

## B3. Count the 2023 rows
Loop over `rows` and count the rows whose year is 2023. Careful: the year is the **text** `"2023"`, not the number `2023`.

In [ ]:
count_2023 = 0
for row in rows:
    if row["year"] == "2023":
        count_2023 = count_2023 + 1
print(count_2023)

In [ ]:
# check
assert count_2023 == 56
print("B3 OK")

## B4. Total 2023 exports
Extend B3: add up `exports_usd_m` for the 2023 rows. Convert each value with `float()` before adding.

In [ ]:
total_2023 = 0.0
for row in rows:
    if row["year"] == "2023":
        total_2023 = total_2023 + float(row["exports_usd_m"])
print(f"{total_2023:,.0f} USD m")

In [ ]:
# check
assert abs(total_2023 - 12269499) < 1
print("B4 OK")

---
# Part C: Reading CSV files with pandas

## C1. One line instead of a loop
`pd.read_csv(path)` reads the whole file into a DataFrame. Load `trade_summary.csv` into `trade`.

In [ ]:
trade = pd.read_csv(DATA / "trade_summary.csv")
trade.head()

In [ ]:
# check
assert trade.shape == (280, 8)
print("C1 OK")

## C2. Column types
`dtypes` shows the type pandas chose for each column. Unlike the csv module, numbers are already numbers.

*Run this cell; no changes needed.*

In [ ]:
trade.dtypes

**Question:** Which columns are numbers (`int64`, `float64`) and which are text (`object`)?

*Your answer:* `year` is int64; `exports_usd_m` and `imports_usd_m` are float64; the rest are text (object).

## C3. Load a second file
Load `tariffs_mfn.csv` into `tariffs` and show its first rows.

In [ ]:
tariffs = pd.read_csv(DATA / "tariffs_mfn.csv")
tariffs.head()

In [ ]:
# check
assert tariffs.shape == (280, 4)
print("C3 OK")

## C4. The same total, the pandas way
This is a preview of Module 04: filter to 2023, pick the exports column and sum it. Compare with your answer from B4.

*Run this cell; no changes needed.*

In [ ]:
trade[trade["year"] == 2023]["exports_usd_m"].sum()

---
# Part D: Reading Excel

## D1. List the sheets
`pd.ExcelFile(path).sheet_names` lists the sheets in a workbook. Store them in `sheets`.

In [ ]:
sheets = pd.ExcelFile(DATA / "countries.xlsx").sheet_names
print(sheets)

In [ ]:
# check
assert sheets == ["countries", "notes"]
print("D1 OK")

## D2. Load one sheet
Load the `countries` sheet into `countries` with `pd.read_excel(path, sheet_name=...)`.

In [ ]:
countries = pd.read_excel(DATA / "countries.xlsx", sheet_name="countries")
countries.head()

In [ ]:
# check
assert len(countries) == 14
print("D2 OK")

---
# Part E: Handling missing and unsupported files

## E1. What an error looks like
`imf_indicators.csv` does not exist. `try` runs code that might fail; `except` catches the error so the notebook carries on.

*Run this cell; no changes needed.*

In [ ]:
try:
    pd.read_csv(DATA / "imf_indicators.csv")
except FileNotFoundError as e:
    print("Caught:", type(e).__name__)

## E2. A safe CSV reader
Write `safe_read_csv(path)`: return `pd.read_csv(path)`, but if a `FileNotFoundError` occurs, print a message and return `None`.

**Example**
```python
try:
    ...
except FileNotFoundError:
    print("...")
    return None
```

In [ ]:
def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        print(f"File not found: {path.name}")
        return None

print(safe_read_csv(DATA / "imf_indicators.csv"))

In [ ]:
# check
assert safe_read_csv(DATA / "imf_indicators.csv") is None
assert len(safe_read_csv(DATA / "tariffs_mfn.csv")) == 280
print("E2 OK")

## E3. CSV or Excel
Write `load_dataset(path)`. Use `path.suffix` (for example `".csv"`) to decide:
- `.csv`: use `safe_read_csv(path)`
- `.xlsx`: use `pd.read_excel(path)`
- anything else: print `Unsupported file type` and return `None`

In [ ]:
def load_dataset(path):
    if path.suffix == ".csv":
        return safe_read_csv(path)
    elif path.suffix == ".xlsx":
        return pd.read_excel(path)
    else:
        print(f"Unsupported file type: {path.name}")
        return None

In [ ]:
# check
assert len(load_dataset(DATA / "countries.xlsx")) == 14
assert load_dataset(Path("README.md")) is None
print("E3 OK")

## E4. Load a list of files
Loop over `files_to_load`. For each name call `load_dataset(DATA / name)`; if the result is not `None`, store it in the dictionary `datasets` under its name.

In [ ]:
files_to_load = ["trade_summary.csv", "tariffs_mfn.csv", "countries.xlsx",
                 "imf_indicators.csv", "README.md"]
datasets = {}
for name in files_to_load:
    df = load_dataset(DATA / name)
    if df is not None:
        datasets[name] = df
print(list(datasets))

In [ ]:
# check
assert set(datasets) == {"trade_summary.csv", "tariffs_mfn.csv", "countries.xlsx"}
print("E4 OK")

## E5. Which files have gaps?
`df.isna().sum().sum()` counts every missing value in a DataFrame. Print the count for each dataset in `datasets`.

In [ ]:
for name, df in datasets.items():
    print(name, df.isna().sum().sum())

**Question:** Which dataset has missing values, and in which column? (Try `datasets["tariffs_mfn.csv"].isna().sum()`.)

*Your answer:* `tariffs_mfn.csv` has missing values in `avg_mfn_tariff_pct`. We decide how to treat gaps in Module 07.

---
# Part F: Writing output files

## F1. Create the output folder
`OUT.mkdir(exist_ok=True)` creates the folder, and does nothing if it already exists.

In [ ]:
OUT.mkdir(exist_ok=True)

In [ ]:
# check
assert OUT.exists()
print("F1 OK")

## F2. Select the rows to save
This filter (covered properly in Module 04) keeps the Africa rows for 2023.

*Run this cell; no changes needed.*

In [ ]:
africa = trade[(trade["region"] == "Africa") & (trade["year"] == 2023)]
africa

## F3. Save as CSV and Excel
Save `africa` as `output/africa_2023.csv` and `output/africa_2023.xlsx`, without the row numbers.

**Example**
```python
df.to_csv(OUT / "file.csv", index=False)
```

In [ ]:
africa.to_csv(OUT / "africa_2023.csv", index=False)
africa.to_excel(OUT / "africa_2023.xlsx", index=False)

In [ ]:
# check
assert (OUT / "africa_2023.csv").exists() and (OUT / "africa_2023.xlsx").exists()
print("F3 OK")

## F4. Write a text report
Write `output/load_report.txt` with one line per dataset, e.g. `trade_summary.csv: 280 rows`. Remember that `write()` does not add a newline: end each line with `\n`.

In [ ]:
with open(OUT / "load_report.txt", "w", encoding="utf-8") as f:
    for name, df in datasets.items():
        f.write(f"{name}: {len(df)} rows\n")
print((OUT / "load_report.txt").read_text())

In [ ]:
# check
assert "trade_summary.csv: 280 rows" in (OUT / "load_report.txt").read_text()
print("F4 OK")